# §66 — Is `kim_nkc` under-weighted? Gradient probe + **L5dfin0** on the CLIP-only Nexar host (s2024)

Rule: `core/docs/final/KIM_RUN.md` **§66**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. PRELIMINARY (n = 1). **Decides nothing; K5f stays the reference.**

| step | what | GPU | estimate |
|---|---|---|---|
| 1-4 | setup, deps, helpers, preflight | no | 3-5 min |
| 5 | tests — **raises if red** | no | 3 min |
| 6 | stage inputs | no | 5-10 min |
| 7 | **NK1 gradient probe** on the saved L5dfi and K5dfirr checkpoints (16 batches each, no training) | yes | 5-15 min |
| 8 | **NK2 train L5dfin0** = L5dfi + `kim.nkc_weight=0` (1,749 steps); L5dfirr first if this GPU is not L5dfi's (NK3) | **yes** | ≈ L5dfi's time (× 2 if L5dfirr) |
| 9 | reads of the new row(s): Nexar val, DoTA-CAP-dev, DADA, DAD | yes | 20-40 min per row |
| 10 | report -> **paste `REPORTS/kim_nkc_report.md` and both `grad_probe/*/kim_grad_probe.md` to Claude** | no | 5 min |

A re-run skips what is on Drive. **Watch the log: past 2 s/step, stop and paste it.** Thesis / Thesis-V2 and every earlier row are only read.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Paths, rows, inputs

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, Nexar caches, A2 bakes, KNN, census
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, earlier rows and reads; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 66. Is `kim_nkc` under-weighted?' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §66 -- re-upload kat-vad/'
for tool in ('kim_grad_probe', 'kim_nkc_report', 'kim_clip_host_report'):
    assert (REPO / 'core/tools' / f'{tool}.py').is_file(), f'no {tool} -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'KIM_NKC_ROW'), 'constants has no §66 names -- re-upload kat-vad/'

K5       = constants.KIM_MAIN_ARM                  # "K5": names run / eval sub-folders only
SEED     = constants.KIM_SEED                      # 2024
OFF      = constants.V2_OFF
IGN      = constants.DVS_ANCHOR_MODE_IGNORE
DATASET  = constants.NEXAR_DATASET
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV  = constants.V2_SPLIT_DOTA_CAP_DEV
STRIDE_B = constants.V2_E1_STRIDE_BC               # DoTA s1[::3] = protocol B rows
KIM_FLAGS = constants.KIM_ABLATION_FLAGS           # the four K5dfi KIM flags
RR       = constants.KIM_ABLATION_RERUN_SUFFIX
NKC_KEY  = constants.KIM_NKC_WEIGHT_KEY
ROW      = constants.KIM_NKC_ROW                   # L5dfin0
ROOT     = constants.OUTPUT_ROOT / constants.KIM_NKC_OUTPUT_DIRNAME     # new rows + REPORTS
REPORTS  = ROOT / 'REPORTS'
V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
NEXAR_SPLIT = constants.V2_SPLIT_NEXAR_VAL
DAD_NAME, DAD_STRIDE = constants.DAD_DATASET, constants.DAD_STRIDE
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}' / constants.KNN_CACHE_FILENAME
CLIP_S8  = V2_CACHE / 'clip' / 'Nexar_s8_ncc'                     # L5dfi's training input (plain CLIP)
A2_NEXAR = V2_CACHE / 'v2' / 'A2_S' / 'Nexar_s8'                  # K5dfirr's training input (probe only)
NEXAR_X  = V2_CACHE / 'clip' / constants.NEXAR_CLIP_S1_DATASET
DAD_DATA = Path(DRIVE_F) / 'data' / f'{DAD_NAME}_s{DAD_STRIDE}'
DAD_PLAIN = constants.CLIP_CACHE_DIR / f'{DAD_NAME}_s{DAD_STRIDE}_ncc'
CTRL_SAVED = constants.OUTPUT_ROOT / constants.KIM_CLIP_HOST_OUTPUT_DIRNAME / constants.KIM_NKC_CONTROL_ROW   # §64 L5dfi
K5_PROBE = constants.OUTPUT_ROOT / constants.KIM_ABLATION_OUTPUT_DIRNAME / constants.KIM_CLIP_HOST_ROWS['nexar']['K5']  # K5dfirr
LOCAL    = Path('/content/kim_nkc')
LOCAL.mkdir(parents=True, exist_ok=True)

INPUTS = [('Nexar_window', WINDOW / constants.LABELS_TRAIN_FILENAME), ('KNN cache', KNN), ('Nexar census', CENSUS),
          ('Nexar plain CLIP s8', CLIP_S8), ('Nexar A2 bake', A2_NEXAR / constants.V2_INPUT_MANIFEST_FILENAME),
          ('Nexar CLIP s1', NEXAR_X), ('DoTA s1', DOTA_S1), ('T2 meta', T2 / constants.META_FILENAME),
          ('T2 census', T2_CENSUS), ('T2 plain CLIP', T2_CLIP), ('DADA annotation', ANNOT), ('DoTA metadata', DOTA / 'metadata_val.json'),
          ('DAD labels', DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME), ('DAD plain CLIP', DAD_PLAIN)]
for root in (CTRL_SAVED, K5_PROBE):
    INPUTS += [(f'{root.name} verified', root / K5 / f's{SEED}' / 'verified.json'),
               (f'{root.name} env', root / K5 / f's{SEED}' / constants.KIM_RUN_ENV_FILENAME),
               (f'{root.name} checkpoint', root / K5 / f's{SEED}' / 'checkpoint_last.pt')]
INPUTS += [(f'L5dfi {s} read', p) for s, p in (
    ('nexar_val', CTRL_SAVED / 'eval' / 'nexar' / CTRL_SAVED.name / 'nexar_readout.json'),
    ('DoTA', CTRL_SAVED / 'eval' / 'dota' / K5 / 'clip_scores.npz'),
    ('DADA', CTRL_SAVED / 'eval' / 'dada' / K5 / 'clip_scores.npz'),
    ('DAD', CTRL_SAVED / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}' / 'results.json'))]
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | control {CTRL_SAVED.name} | probe rows {CTRL_SAVED.name}, {K5_PROBE.name} | new runs -> {ROOT}')

## 3. Dependencies

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- probe, training and reads need one'
df -h /content | tail -1

## 4. Helpers + preflight (CPU)

In [ ]:
import torch  # noqa: E402
import yaml  # noqa: E402

from core.config import Config                         # noqa: E402
from core.data.v2_splits import load_split             # noqa: E402
from core.tools import k5_edge_report as edge          # noqa: E402
from core.tools import kim_ablation_report as ka       # noqa: E402
from core.tools.dada_whole_eval import test_sources    # noqa: E402
from core.tools.v5_stage1 import _flat                 # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    """Stream a child's output here (and to ``log``); raise with its tail on failure."""
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    """Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2)."""
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    names = [f.name for f in src.iterdir() if f.is_file()] if ids is None else \
        [f'{v}.npy' for v in ids] + [m.name for p in ('*.json', '*.npz') for m in src.glob(p)]
    todo = [n for n in names if not (dst / n).exists()]
    with ThreadPoolExecutor(16) as pool:
        list(pool.map(lambda n: shutil.copy2(src / n, dst / n), todo))
    if todo:
        print(f'  staged {src.name}: {len(todo)} files in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    """VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10)."""
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    """The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step)."""
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def cfg_of(root):
    return _flat(yaml.safe_load((root / K5 / f's{SEED}' / 'config.yaml').read_text()))


def run_dir(root):
    return root / K5 / f's{SEED}'


def last_epoch(rows, key):
    last = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    return sum(r.get(key, float('nan')) for r in last) / len(last)


def check_run(root, kind):
    """J10, §31 floor, CH9 config diff to the saved L0. ``kind`` in {'L5', 'L5cap', 'L0rr'}."""
    rd = run_dir(root)
    rows = metrics_rows(rd / 'metrics.jsonl')
    assert ckpt_step(rd / 'checkpoint_last.pt') == rows[-1]['global_step'] == STEPS, f'{root.name}: J10'
    got = cfg_of(root)
    if kind == 'L5':
        print(f'{root.name}: KIM flags vs {L0_SAVED.name}:', ka.check_kim_only(got, l0_cfg, root.name))
    elif kind == 'L5cap':
        diff = ka.config_diff(l0_cfg, got)
        assert diff == {k: (False, True) for k in CAP_FLAGS}, f'{root.name}: diff {diff} (§64 CH9)'
        print(f'{root.name}: Kcap flags vs {L0_SAVED.name}:', sorted(diff))
    else:
        diff = ka.config_diff(l0_cfg, got)
        assert not diff, f'{root.name}: differs from {L0_SAVED.name} by {diff} (a re-run allows nothing)'
    mul = last_epoch(rows, 'mul_mil')
    assert mul > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{root.name}: last-epoch mul_mil {mul:.4f} -- paste to Claude'
    keys = ('mil', 'mul_mil', 'kim_mil', 'kim_nkc')
    print('  last epoch: ' + ' | '.join(f'{k} {last_epoch(rows, k):.4f}' for k in keys))


def train_row(name, sets, kind):
    """Train one row into ROOT/<name> (resumable, atomic publish, J10, env recorded)."""
    root = ROOT / name
    rd, ckpt = run_dir(root), run_dir(root) / 'checkpoint_last.pt'
    if is_verified(rd, STEPS):
        print(f'{name} s{SEED}: verified on Drive -- not retrained')
        check_run(root, kind)
        return root
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / name
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    env = {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': str(torch.version.cuda)}
    print(f'{name}: trains on {env}')
    (local / constants.KIM_RUN_ENV_FILENAME).write_text(json.dumps(env))
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in sets for x in ('--set', s)], '--data-dir', L_DATA,
         '--clip-dir', L_TRAIN_X, '--output-dir', local, '--knn-cache', L_KNN, *resume], log=LOCAL / f'train_{name}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{name}: J10 locally {last["global_step"]}'
    for f in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml', constants.KIM_RUN_ENV_FILENAME):
        publish(local / f, rd / f)
    flush_drive()
    assert ckpt_step(ckpt) == STEPS, f'{name}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': ckpt.stat().st_size}))
    publish(local / 'verified.json', rd / 'verified.json')
    print(f'{name}: trained in {(time.time() - t0) / 60:.0f} min | step-1 mil {metrics_rows(local / "metrics.jsonl")[0]["mil"]:.6f}')
    check_run(root, kind)
    return root


def start(ctrl_root, new_root):
    st = edge.start_check(run_dir(ctrl_root), run_dir(new_root))
    print(f"CH3 step 1: {new_root.name} {st['new']:.6f} vs {ctrl_root.name} {st['control']:.6f} -> "
          f"{'matched' if st['same_draws'] else 'DIFFERENT START'}")
    return st['same_draws']


def read_dota(root):
    """DoTA-CAP-dev, protocol B on plain DoTA_s1_ncc[::3] (every L row)."""
    out = root / 'eval' / 'dota' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / root.name / 'dota'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', run_dir(root) / 'checkpoint_last.pt',
             '--input-dir', L_DOTA_S1, '--s1-dir', L_DOTA_S1, '--metadata', DOTA / 'metadata_val.json',
             '--split-file', DOTA / 'val_split.txt', '--data-dir', L_LAB, '--split', CAP_DEV, '--stride', STRIDE_B,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{root.name}_dota.log')
        publish(local, out)
    r = json.loads((out / 'protocol_b_readout.json').read_text())
    assert r['clips'] == len(CAP_IDS) and r['global_steps'] == {f's{SEED}': STEPS} and r['arm']['motion'] == OFF, r
    print(f"{root.name} DoTA-CAP-dev: macro {r['macro']['mean']:.4f}")


def read_dada(root):
    """DADA whole on plain DADA2000_orig CLIP (every L row)."""
    out = root / 'eval' / 'dada' / K5
    if not (out / 'clip_aucs.json').exists():
        local = LOCAL / 'eval' / root.name / 'dada'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', run_dir(root) / 'checkpoint_last.pt',
             '--t2-dir', L_T2, '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', L_T2CLIP,
             '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{root.name}_dada.log')
        publish(local, out)
    r = json.loads((out / 'dada_readout.json').read_text())
    assert r['videos'] == len(DADA_IDS) and r['global_steps'] == {f's{SEED}': STEPS} and r['arm']['motion'] == OFF, r
    print(f"{root.name} DADA whole: macro {r['macro']['mean']:.4f}")


def check_run(root, kind):
    """J10, §31 floor, §66 NK4 config diff to the saved L5dfi. ``kind`` in {'n0', 'rr'}."""
    rd = run_dir(root)
    rows = metrics_rows(rd / 'metrics.jsonl')
    assert ckpt_step(rd / 'checkpoint_last.pt') == rows[-1]['global_step'] == STEPS, f'{root.name}: J10'
    diff = ka.config_diff(ctrl_cfg, cfg_of(root))
    want = {NKC_KEY: (ctrl_cfg[NKC_KEY], constants.KIM_NKC_OFF_WEIGHT)} if kind == 'n0' else {}
    assert diff == want, f'{root.name}: diff {diff}, §66 NK4 allows {want}'
    mul = last_epoch(rows, 'mul_mil')
    assert mul > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{root.name}: last-epoch mul_mil {mul:.4f} -- paste to Claude'
    keys = ('mil', 'mul_mil', 'kim_mil', 'kim_nkc')
    print(f'{root.name} ({kind}) diff {diff} | last epoch: ' + ' | '.join(f'{k} {last_epoch(rows, k):.4f}' for k in keys))


def this_env():
    return {'gpu': torch.cuda.get_device_name(0), 'torch': torch.__version__, 'cuda': str(torch.version.cuda)}


# Preflight on small files only (C39)
ctrl_cfg = cfg_of(CTRL_SAVED)
assert ctrl_cfg['kim.enabled'] and ctrl_cfg['kim.prototypes'] and ctrl_cfg['v2.motion'] == OFF and ctrl_cfg['model.motion_dim'] == 0, 'CTRL_SAVED is not L5dfi'
assert ctrl_cfg[NKC_KEY] > 0 and (ctrl_cfg['dvs.theta'], ctrl_cfg['loss.dvs_anchor_mode'], ctrl_cfg['data.dataset']) == (0.7, IGN, DATASET), 'CTRL_SAVED is not L5dfi'
k5_cfg = cfg_of(K5_PROBE)
assert k5_cfg['kim.enabled'] and k5_cfg['kim.prototypes'] and k5_cfg['data.dataset'] == DATASET and k5_cfg[NKC_KEY] == ctrl_cfg[NKC_KEY], 'K5_PROBE is not K5dfirr'
STEPS = json.loads((run_dir(CTRL_SAVED) / 'verified.json').read_text())['step']     # 1,749
EPOCHS = ctrl_cfg['train.num_epochs']                                              # 33
assert is_verified(run_dir(CTRL_SAVED), STEPS) and metrics_rows(run_dir(CTRL_SAVED) / 'metrics.jsonl')[-1]['global_step'] == STEPS, 'L5dfi: J10'
assert EPOCHS - 1 == constants.KIM_GRAD_PROBE_EPOCH, f'probe epoch {constants.KIM_GRAD_PROBE_EPOCH} is not the last of {EPOCHS}'
CTRL_ENV = json.loads((run_dir(CTRL_SAVED) / constants.KIM_RUN_ENV_FILENAME).read_text())
TRAIN_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_TRAIN))
DADA_IDS, CAP_IDS = test_sources(T2), sorted(load_split(CAP_DEV))
NEXAR_IDS = sorted(load_split(NEXAR_SPLIT))
DAD_IDS = sorted(json.loads((DAD_DATA / constants.FRAME_LABELS_TEST_FILENAME).read_text()))
assert len(DADA_IDS) == 370 and len(DAD_IDS) == constants.DAD_TEST_VIDEOS and len(TRAIN_IDS) == 900
print(f'preflight OK: L5dfi step {STEPS} ({EPOCHS} epochs), {NKC_KEY} {ctrl_cfg[NKC_KEY]}, trained on {CTRL_ENV} | '
      f'{NEXAR_SPLIT} {len(NEXAR_IDS)}, DoTA-CAP-dev {len(CAP_IDS)}, DADA {len(DADA_IDS)}, DAD {len(DAD_IDS)}')

## 5. Tests on the touched code (CPU) — **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_kim_grad_probe.py',
     'core/tests/test_kim_nkc_report.py', 'core/tests/test_kim_clip_host_report.py', 'core/tests/test_kim.py'])

## 6. Stage inputs

In [ ]:
L_DATA    = stage(WINDOW, LOCAL / 'stage' / 'data')
L_TRAIN_X = stage(CLIP_S8, LOCAL / 'stage' / 'nexar_s8', TRAIN_IDS)
L_A2      = stage(A2_NEXAR, LOCAL / 'stage' / 'a2_nexar', TRAIN_IDS)        # K5dfirr's probe input only
L_KNN     = stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name)
L_DOTA_S1 = stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS)
L_T2CLIP  = stage(T2_CLIP, LOCAL / 'stage' / 'dada_clip', DADA_IDS)
L_NEXAR_X = stage(NEXAR_X, LOCAL / 'stage' / 'nexar_x', NEXAR_IDS)
L_DAD     = stage(DAD_PLAIN, LOCAL / 'stage' / 'dad_plain', DAD_IDS)
L_DADMETA = stage(DAD_DATA, LOCAL / 'stage' / 'dad_meta')
L_LAB     = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)
print('staged')

## 7. NK1 gradient probe (printed, decides nothing)

Each saved checkpoint with its own `config.yaml` and training input: L5dfi on plain `Nexar_s8_ncc`, K5dfirr on the A2 bake. 16 batches of 64 from epoch 32's order, fp32, no optimizer step, prototypes restored.

In [ ]:
assert torch.cuda.is_available(), 'the probe runs on the GPU (fp32, 16 batches of 64)'
PROBES = {CTRL_SAVED.name: (CTRL_SAVED, L_TRAIN_X), K5_PROBE.name: (K5_PROBE, L_A2)}
for name, (root, clip_dir) in PROBES.items():
    out = REPORTS / 'grad_probe' / name
    stem = constants.KIM_GRAD_PROBE_STEM
    if not (out / f'{stem}.json').exists():
        local = LOCAL / 'grad_probe' / name
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.kim_grad_probe', '--label', name, '--config', run_dir(root) / 'config.yaml',
             '--checkpoint', run_dir(root) / 'checkpoint_last.pt', '--data-dir', L_DATA, '--clip-dir', clip_dir,
             '--knn-cache', L_KNN, '--device', 'cuda', '--output-dir', local], log=LOCAL / f'probe_{name}.log')
        publish(local, out)
    print((out / f'{stem}.md').read_text())
flush_drive()

## 8. NK2 / NK3: train **L5dfin0**; the control is L5dfi only on L5dfi's own GPU / torch / CUDA

L5dfi's exact command (§64 CH1) + `kim.nkc_weight=0`. If this session's environment differs from L5dfi's `env.json`, L5dfi's exact command is re-trained first as **L5dfirr** and becomes the control (all its reads are done here). Step-1 `mil` must match (NK4).

In [ ]:
# L5dfi's exact command (§36 R1 + §64 CH1)
L5_SETS = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
           f'data.dataset={DATASET}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
           'kip.enabled=false', f'loss.dvs_anchor_mode={IGN}', 'data.is_egocentric=false',
           f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}', 'dvs.theta=0.7', 'dvs.theta_ego=0.85',
           f'v2.motion={OFF}', 'model.motion_dim=0', *[f'{k}=true' for k in KIM_FLAGS]]
assert torch.cuda.is_available(), 'torch sees no GPU'
env = this_env()
same_env = env == CTRL_ENV
print(f'this session {env} | L5dfi trained on {CTRL_ENV} -> control = {"L5dfi (saved)" if same_env else "L5dfirr (re-trained here)"}')
if same_env:
    CTRL = CTRL_SAVED
else:
    CTRL = train_row(CTRL_SAVED.name + RR, L5_SETS, 'rr')
NEW = train_row(ROW, [*L5_SETS, f'{NKC_KEY}={constants.KIM_NKC_OFF_WEIGHT}'], 'n0')
assert json.loads((run_dir(NEW) / constants.KIM_RUN_ENV_FILENAME).read_text()) == json.loads((run_dir(CTRL) / constants.KIM_RUN_ENV_FILENAME).read_text()), \
    'control and L5dfin0 trained on different environments -- §66 NK3, paste to Claude'
assert start(CTRL, NEW), 'UNMATCHED START (§66 NK4) -- paste to Claude'
print('rows:', CTRL.name, NEW.name)

## 9. Reads of the new row(s): Nexar val (`--save-scores`), DoTA-CAP-dev (protocol B), DADA whole, DAD — plain CLIP inputs

In [ ]:
from core.tools.nexar_eval import ITEM_AUCS_JSON, READOUT_JSON, VIDEO_STATS_JSON  # noqa: E402


def read_nexar(root):
    """nexar_val with --save-scores; plain rows: no --stats-dir, no --video-s1-dir."""
    out = root / 'eval' / 'nexar' / root.name
    done = all((out / f).exists() for f in (READOUT_JSON, ITEM_AUCS_JSON, VIDEO_STATS_JSON)) and \
        len(list((out / 'scores').glob('*.npz'))) == len(NEXAR_IDS)
    if not done:
        local = LOCAL / 'eval' / root.name / 'nexar'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.tools.nexar_eval', '--run', f's{SEED}', run_dir(root) / 'checkpoint_last.pt',
             '--census', CENSUS, '--clip-s1-dir', L_NEXAR_X, '--data-dir', L_DEFS, '--split', NEXAR_SPLIT,
             '--save-scores', '--out-dir', local], log=LOCAL / f'{root.name}_nexar.log')
        publish(local, out)
    r = json.loads((out / READOUT_JSON).read_text())
    assert r['split'] == NEXAR_SPLIT and r['videos'] == len(NEXAR_IDS) and r['global_steps'] == {f's{SEED}': STEPS}, r['global_steps']
    print(f"{root.name} nexar_val: crops macro {r['crops']['mean_over_placements']['macro']['mean']:.4f} | whole macro {r['whole']['macro']['mean']:.4f}")


def read_dad(root):
    out = root / 'eval' / constants.DAD_OUTPUT_SET / K5 / f's{SEED}'
    if not (out / 'results.json').exists():
        local = LOCAL / 'eval' / root.name / 'dad'
        shutil.rmtree(local, ignore_errors=True)
        run([sys.executable, '-m', 'core.evaluate', '--ckpt', run_dir(root) / 'checkpoint_last.pt', '--set', f'data.dataset={DAD_NAME}',
             '--data-dir', L_DADMETA, '--clip-dir', L_DAD, '--score-norm', 'auto', '--save-scores', '--output-dir', local],
            log=LOCAL / f'{root.name}_dad.log')
        publish(local, out)
    r = json.loads((out / 'results.json').read_text())
    assert r['score_norm'] == 'none' and r['num_videos'] == len(DAD_IDS), r
    print(f"{root.name} DAD: macro {r['auc_macro']:.4f} | micro {r['auc']:.4f}")


for root in ([NEW] if CTRL == CTRL_SAVED else [CTRL, NEW]):
    read_nexar(root)
    read_dota(root)
    read_dada(root)
    read_dad(root)
    flush_drive()

## 10. Report (§66 NK4–NK7) -> **paste `REPORTS/kim_nkc_report.md` and the two `grad_probe/*/kim_grad_probe.md` to Claude**

In [ ]:
local = LOCAL / 'report'
shutil.rmtree(local, ignore_errors=True)
local.mkdir(parents=True)
run([sys.executable, '-m', 'core.tools.kim_nkc_report', '--control', CTRL.name, CTRL, '--row', NEW.name, NEW,
     '--seed', SEED, '--out-dir', local])
publish(local, REPORTS)
print((REPORTS / f'{constants.KIM_NKC_REPORT_STEM}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print('Drive flushed -- paste the report and both probes to Claude')